In [0]:
import pandas as pd

In [0]:
%sql
create catalog if not exists 'heathcare-metrics-project' 

In [0]:
%sql
create schema if not exists `heathcare-metrics-project`.`source`

### Uploading master CSV as an external volume


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS `heathcare-metrics-project`.source.`heathcare_metrics_volume`

In [0]:
df = (spark
      .read
      .format("csv")
      .option("header", "True")
      .option("inferSchema", "True")
      .load("/Volumes/heathcare-metrics-project/source/heathcare_metrics_volume/PBJ_Daily_Nurse_Staffing_Q2_2024.csv")
)

df.printSchema()

In [0]:
df.columns

In [0]:
len(df.columns)

In [0]:
type(df.columns)

In [0]:
df.display()

In [0]:
duplicates = df.groupBy(df.columns).count().filter("count > 1")

In [0]:
print(duplicates)

In [0]:
duplicates.display()

In [0]:
df = df.drop_duplicates()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count null values for each column
null_counts = df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
print(null_counts.collect()[0].asDict())

In [0]:
from pyspark.sql.functions import col
import re

def sanitize_column_name(name):
    # Replace invalid characters with underscores
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

df_clean = df.select([col(c).alias(sanitize_column_name(c)) for c in df.columns])

df_clean.write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("`heathcare-metrics-project`.source.PBJ_Daily_Nurse_Staffing_Q2_2024")


In [0]:
%sql
select * from `heathcare-metrics-project`.source.pbj_daily_nurse_staffing_q2_2024

### Uploading supporting files from google drive


In [0]:
%pip install gdown

In [0]:
dbutils.library.restartPython()

In [0]:
import gdown
import shutil

In [0]:

file_id = "1gsofjXa-DHRPgPw0iZQa3cl4VqVP74jb"

url = f"https://drive.google.com/uc?id={file_id}"

local_path = "/tmp/NH_ProviderInfo_Oct2024.csv"
volume_path = "/Volumes/heathcare-metrics-project/source/heathcare_metrics_volume/NH_ProviderInfo_Oct2024.csv"

# Download the file from Google Drive
gdown.download(url, local_path, quiet=False)

# Copy to volume so Spark can read it
shutil.copy(local_path, volume_path)

df1 = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(volume_path)
)


In [0]:
# Sanitize column names for Delta Lake
from pyspark.sql.functions import col
import re

def sanitize_column_name(name):
    # Replace invalid characters with underscores
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

df1_clean = df1.select([col(c).alias(sanitize_column_name(c)) for c in df1.columns])

df1_clean.write \
  .mode("overwrite") \
  .saveAsTable("`heathcare-metrics-project`.source.nh_provider_info_oct2024")

In [0]:
df1.display()

In [0]:
file_id = "1yxljFr7-LvdDb_P9F1MF3ay4WMBHOzqS"

url = f"https://drive.google.com/uc?id={file_id}"

local_path = "/tmp/NH_StateUSAverages_Oct2024.csv"
volume_path = "/Volumes/heathcare-metrics-project/source/heathcare_metrics_volume/NH_StateUSAverages_Oct2024.csv"

# Download the file from Google Drive
gdown.download(url, local_path, quiet=False)

# Copy to volume so Spark can read it
shutil.copy(local_path, volume_path)

df2 = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(volume_path)
)

def sanitize_column_name(name):
    # Replace invalid characters with underscores
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

df2_clean = df1.select([col(c).alias(sanitize_column_name(c)) for c in df1.columns])

df2_clean.write \
  .mode("overwrite") \
  .saveAsTable("`heathcare-metrics-project`.source.nh_stateUSaverages_oct2024")

In [0]:
file_id = "1DlTQXPB53nGwTjPVDApVYnPViJLHqVp2"

url = f"https://drive.google.com/uc?id={file_id}"

local_path = "/tmp/NH_SurveyDates_Oct2024.csv"
volume_path = "/Volumes/heathcare-metrics-project/source/heathcare_metrics_volume/NH_SurveyDates_Oct2024.csv"

# Download the file from Google Drive
gdown.download(url, local_path, quiet=False)

# Copy to volume so Spark can read it
shutil.copy(local_path, volume_path)

df3 = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(volume_path)
)

def sanitize_column_name(name):
    # Replace invalid characters with underscores
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

df3_clean = df1.select([col(c).alias(sanitize_column_name(c)) for c in df1.columns])

df3_clean.write \
  .mode("overwrite") \
  .saveAsTable("`heathcare-metrics-project`.source.nh_surveydates_oct2024")


### Exploring each supporting table
##### source.nh_provider_info_oct2024


In [0]:
%sql
select * from  `heathcare-metrics-project`.source.nh_provider_info_oct2024

In [0]:
duplicates = df1.groupBy(df1.columns).count().filter("count > 1")

In [0]:
duplicates.display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count null values for each column
null_counts = df1.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df1.columns])
print(null_counts.collect()[0].asDict())

In [0]:
df1 = spark.table("`heathcare-metrics-project`.source.nh_provider_info_oct2024")

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Count nulls in each column
null_counts = df1.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df1.columns
]).collect()[0].asDict()

# Convert dictionary to a DataFrame
null_df1 = spark.createDataFrame(
    [Row(column_name=k, null_count=v) for k, v in null_counts.items()]
)

display(null_df1)

In [0]:
null_df1.write \
    .mode("overwrite") \
    .saveAsTable("`heathcare-metrics-project`.source.nh_providerinfo_null_summary")

In [0]:
%sql
SELECT *
FROM `heathcare-metrics-project`.source.nh_providerinfo_null_summary
ORDER BY null_count DESC;

##### source.nh_stateusaverages_oct2024

In [0]:
%sql
select * from `heathcare-metrics-project`.source.nh_stateusaverages_oct2024

In [0]:
df2 = spark.table("`heathcare-metrics-project`.source.nh_stateusaverages_oct2024")
df2.display()

In [0]:
duplicates = df2.groupBy(df2.columns).count().filter("count > 1")

In [0]:
duplicates.display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count null values for each column
null_counts = df2.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df2.columns])
print(null_counts.collect()[0].asDict())

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Count nulls in each column
null_counts = df2.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df2.columns
]).collect()[0].asDict()

# Convert dictionary to a DataFrame
null_df2 = spark.createDataFrame(
    [Row(column_name=k, null_count=v) for k, v in null_counts.items()]
)

display(null_df2)

In [0]:
null_df2.write \
    .mode("overwrite") \
    .saveAsTable("`heathcare-metrics-project`.source.nh_stateusaverages_null_summary")

In [0]:
%sql
select * from `heathcare-metrics-project`.source.nh_stateusaverages_null_summary

##### source.nh_surveydates_oct2024

In [0]:
%sql
select * from `heathcare-metrics-project`.source.nh_surveydates_oct2024

In [0]:
df3 = spark.table("`heathcare-metrics-project`.source.nh_surveydates_oct2024")
df3.display()


In [0]:
# Count null values for each column
null_counts = df3.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df3.columns])
print(null_counts.collect()[0].asDict())

In [0]:
df3 = spark.table("`heathcare-metrics-project`.source.nh_surveydates_oct2024")

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Count nulls in each column
null_counts = df3.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df3.columns
]).collect()[0].asDict()

# Convert dictionary to a DataFrame
null_df3 = spark.createDataFrame(
    [Row(column_name=k, null_count=v) for k, v in null_counts.items()]
)

display(null_df3)

In [0]:
null_df3.write \
    .mode("overwrite") \
    .saveAsTable("`heathcare-metrics-project`.source.nh_surveydates_null_summary")

In [0]:
%sql
select * from `heathcare-metrics-project`.source.nh_surveydates_null_summary